## **Dự đoán tình trạng máy móc thiết bị công nghiệp bị khi đưa vào vận hành**.   

### **Data processing** 

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

# Hỗ trợ chạy notebook từ repo root hoặc trực tiếp trong thư mục classification.
working_dir = Path.cwd().resolve()
repo_candidates = (working_dir, *working_dir.parents)
REPO_ROOT = next((
    path for path in repo_candidates
    if (path / "classification" / "lightgbm_classification.py").is_file()
), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy repo root chứa classification/lightgbm_classification.py."
    )

repo_root_text = str(REPO_ROOT)
if repo_root_text not in sys.path:
    sys.path.insert(0, repo_root_text)

from classification.lightgbm_classification import LightGBMClassification
from classification.evaluation.run_machine_failure_evaluation import (
    DEFAULT_OUTPUT_DIR,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    evaluate_machine_failure_splits,
)


In [ ]:
# Đọc dữ liệu 
DATA_PATH = REPO_ROOT / "classification" / "data" / "raw" / "machine_fail.csv"
df = pd.read_csv(DATA_PATH)
print ('Kích thước dataset', df.shape ) 

df.head ( 10 )

In [ ]:
# Kiểm tra kiểu dữ liệu từng feature  
df.dtypes  

In [ ]:
# Dữ liệu thiếu 
df.isnull ( ).sum ( )

In [ ]:
# Chuẩn hóa dạng số của type of machine 
print ( df['Type'].value_counts ( ))   

df["Type"] = df["Type"].map({
    "L": 0,
    "M": 1,
    "H": 2
}) 

# Sau chuẩn hóa  
df.head ( )

In [ ]:
# Chia dữ liệu thành 2 tập train và test 
X_features = df.loc[:, FEATURE_COLUMNS].copy()

# Nhãn cần dự đoán
y_label = df[TARGET_COLUMN] 

# Chia 80% train - 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_label,
    test_size=0.2,
    random_state=42,
    stratify=y_label
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

### **Training model** 

In [ ]:
# Tiến hành gọi thuật toán xây dựng và training model 
model_lightGBM_cls = LightGBMClassification(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=15,
    max_depth=5,
    random_state=42
) 

print (LightGBMClassification)

In [ ]:
# Training model  
model_lightGBM_cls.fit ( X_train , y_train )

###   **Model Performance Evaluation** 

In [ ]:
# Helper chung tự tạo predict/proba một lần cho cả train và test.
# Mỗi split có artifact riêng và root manifest chỉ cập nhật khi cả hai thành công.
EVALUATION_OUTPUT_ROOT = DEFAULT_OUTPUT_DIR
evaluation_run = evaluate_machine_failure_splits(
    model=model_lightGBM_cls,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    output_dir=EVALUATION_OUTPUT_ROOT,
)

evaluation_results = evaluation_run["split_results"]
PRIMARY_SPLIT = evaluation_run["primary_reporting_split"]
primary_evaluation_result = evaluation_results[PRIMARY_SPLIT]
pipeline_manifest_path = evaluation_run["pipeline_manifest_path"]
print(f"Đã lưu train evaluation tại: {EVALUATION_OUTPUT_ROOT / 'train'}")
print(f"Đã lưu test evaluation (primary) tại: {EVALUATION_OUTPUT_ROOT / 'test'}")
print(f"Root manifest: {pipeline_manifest_path}")
primary_evaluation_result["classification_report"]

In [ ]:
# Bảng so sánh train/test; cột test_primary là kết quả báo cáo chính.
COMPARISON_METRICS = ("accuracy", "precision", "recall", "f1_score", "roc_auc")
metrics_comparison = pd.DataFrame(
    {
        "train": [
            evaluation_results["train"]["metrics"][metric_name]
            for metric_name in COMPARISON_METRICS
        ],
        "test_primary": [
            evaluation_results[PRIMARY_SPLIT]["metrics"][metric_name]
            for metric_name in COMPARISON_METRICS
        ],
    },
    index=pd.Index(COMPARISON_METRICS, name="metric"),
)
metrics_comparison